<a href="https://colab.research.google.com/github/kganeshbabu1998-star/Credit-Card-Approval-Prediction/blob/main/Data_Cleaning_and_Merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np


def data_cleaning(applicant_df, credit_df):

    df = applicant_df.copy()

    if "CNT_FAM_MEMBERS" in df.columns and "CNT_CHILDREN" in df.columns:
        df["FAMILY_DEPENDENCY"] = df["CNT_FAM_MEMBERS"] + df["CNT_CHILDREN"]

    remove_columns = [
        "FLAG_MOBIL",
        "FLAG_WORK_PHONE",
        "FLAG_PHONE",
        "FLAG_EMAIL",
        "DAYS_REGISTRATION",
        "DAYS_ID_PUBLISH"
    ]

    df.drop(
        columns=[col for col in remove_columns if col in df.columns],
        inplace=True
    )

    if "DAYS_BIRTH" in df.columns:
        df["DAYS_BIRTH"] = df["DAYS_BIRTH"].abs()

    if "DAYS_EMPLOYED" in df.columns:
        df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].abs()

    housing_map = {
        "house / apartment": 0,
        "with parents": 1,
        "municipal apartment": 2,
        "rented apartment": 3,
        "office apartment": 4,
        "co-op apartment": 5
    }

    income_map = {
        "Working": 0,
        "Commercial associate": 1,
        "Pensioner": 2,
        "State servant": 3,
        "Student": 4
    }

    education_map = {
        "Lower secondary": 0,
        "Secondary / secondary special": 1,
        "Incomplete higher": 2,
        "Higher education": 3,
        "Academic degree": 4
    }

    family_map = {
        "Single / not married": 0,
        "Married": 1,
        "Civil marriage": 2,
        "Separated": 3,
        "Widow": 4
    }

    if "NAME_HOUSING_TYPE" in df.columns:
        df["NAME_HOUSING_TYPE"] = df["NAME_HOUSING_TYPE"].map(housing_map)

    if "NAME_INCOME_TYPE" in df.columns:
        df["NAME_INCOME_TYPE"] = df["NAME_INCOME_TYPE"].map(income_map)

    if "NAME_EDUCATION_TYPE" in df.columns:
        df["NAME_EDUCATION_TYPE"] = df["NAME_EDUCATION_TYPE"].map(education_map)

    if "NAME_FAMILY_STATUS" in df.columns:
        df["NAME_FAMILY_STATUS"] = df["NAME_FAMILY_STATUS"].map(family_map)

    df.fillna(0, inplace=True)



    credit = credit_df.copy()

    credit_grouped = credit.groupby("ID")

    credit_features = pd.DataFrame()

    credit_features["open_month"] = credit_grouped["MONTHS_BALANCE"].min()
    credit_features["end_months"] = credit_grouped["MONTHS_BALANCE"].max()

    credit_features["window"] = (
        credit_features["end_months"] -
        credit_features["open_month"]
    )

    def payment_status(status):
        status = status.astype(str)

        if any(status.isin(["0"])):
            return 1

        elif any(status.isin(["1", "2", "3", "4", "5"])):
            return 2
        else:
            return 0

    credit_features["PAYMENT_STATUS"] = (
        credit_grouped["STATUS"]
        .apply(payment_status)
    )


    credit_features.reset_index(inplace=True)

    return df, credit_features




In [16]:
applicant_data = pd.read_csv("/content/application_record.csv")
credit_data = pd.read_csv("/content/credit_record.csv")

In [17]:
cleaned_applicant, cleaned_credit = data_cleaning(
    applicant_data,
    credit_data
)

In [18]:
display(cleaned_applicant.head())

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,OCCUPATION_TYPE,CNT_FAM_MEMBERS,FAMILY_DEPENDENCY
0,5008804,M,Y,Y,0,427500.0,0,3,2,0.0,12005,4542,0,2.0,2.0
1,5008805,M,Y,Y,0,427500.0,0,3,2,0.0,12005,4542,0,2.0,2.0
2,5008806,M,Y,Y,0,112500.0,0,1,1,0.0,21474,1134,Security staff,2.0,2.0
3,5008808,F,N,Y,0,270000.0,1,1,0,0.0,19110,3051,Sales staff,1.0,1.0
4,5008809,F,N,Y,0,270000.0,1,1,0,0.0,19110,3051,Sales staff,1.0,1.0


In [19]:
display(cleaned_credit.head())

,ID,open_month,end_months,window,PAYMENT_STATUS
0,5001711,-3,0,3,1
1,5001712,-18,0,18,1
2,5001713,-21,0,21,0
3,5001714,-14,0,14,0
4,5001715,-59,0,59,0
